# A PINNs approach for the computation of eigenvalues in elliptic problems

**Fernández Bonder & Salort**

This notebook reproduces all experiments from the paper using the `pinn_eigen`
module, which centralizes the neural network, loss, training, and plotting logic.
Each example below configures only what is specific to that problem
(domain, potential, hyperparameters).

## Setup

In [ ]:
# If running on Colab, upload pinn_eigen.py first (or clone the repo).
# !git clone https://github.com/amsalort/PINN-eigenvalues
# import sys; sys.path.insert(0, 'PINN-eigenvalues')

import torch
import numpy as np
from functools import partial
import pinn_eigen as pe

torch.manual_seed(0)   # reproducibility

---
## Example 1 — Harmonic potential in the 2-D unit disk

$$-\Delta u + \frac{\omega^2}{2}|x|^2 u = E u \quad \text{in } D, \qquad u=0 \text{ on } \partial D$$

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
E_min, E_max, n_steps = 0.0, 55.0, 80
omega = 1.0

sampler    = partial(pe.sample_disk, radius=1.0)
V_fn       = partial(pe.V_harmonic, omega=omega)
model_fn   = lambda: pe.EigenFunctionNN(dim=2, boundary_fn=pe.boundary_disk)

E_values   = np.linspace(E_min, E_max, n_steps)
mu         = 100.0

# ── Sweep ──────────────────────────────────────────────────────────────────
residuals, models = pe.sweep(
    model_fn, E_values, sampler, V_fn,
    mu=mu, reset_every=50, n_epochs=3000,
)

# ── Loss curve ─────────────────────────────────────────────────────────────
pe.plot_loss_curve(E_values, residuals, title="Ex 1: Harmonic potential, unit disk")

# ── Refine and print eigenvalues ───────────────────────────────────────────
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn, threshold=0.3)

# ── Plot eigenfunctions ────────────────────────────────────────────────────
for (E_approx, _), i in zip(eigenvalues, idx):
    pe.plot_eigenfunction_disk(models[i], E_val=E_approx)

---
## Example 2 — Laplacian in the 2-D unit square

$$-\Delta u = E u \quad \text{in } [0,1]^2, \qquad u=0 \text{ on } \partial[0,1]^2$$

Exact eigenvalues: $E_{mn} = \pi^2(m^2+n^2)$.

In [ ]:
EXACT = [2*np.pi**2, 5*np.pi**2, 8*np.pi**2, 10*np.pi**2, 13*np.pi**2]  # first few

E_min, E_max, n_steps = 0.0, 105.0, 80
sampler  = partial(pe.sample_square, x0=0.0, x1=1.0)
V_fn     = pe.V_zero
model_fn = lambda: pe.EigenFunctionNN(dim=2, boundary_fn=pe.boundary_square)

E_values = np.linspace(E_min, E_max, n_steps)
mu       = 500.0

residuals, models = pe.sweep(
    model_fn, E_values, sampler, V_fn,
    mu=mu, reset_every=25, n_epochs=5000,
)

pe.plot_loss_curve(E_values, residuals, exact_eigenvalues=EXACT,
                   title="Ex 2: Laplacian, unit square")
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn, threshold=2.0)

for (E_approx, _), i in zip(eigenvalues, idx):
    pe.plot_eigenfunction_square(models[i], E_val=E_approx)

---
## Example 3 — p-Laplacian in the 2-D unit disk  (p = 2.2)

$$-\nabla\cdot(|\nabla u|^{p-2}\nabla u) = E u \quad \text{in } D, \qquad u=0 \text{ on } \partial D$$

In [ ]:
p = 2.2
E_min, E_max, n_steps = 0.0, 50.0, 50

sampler  = partial(pe.sample_disk, radius=1.0)
V_fn     = pe.V_zero
model_fn = lambda: pe.EigenFunctionNN(dim=2, boundary_fn=pe.boundary_disk)

E_values = np.linspace(E_min, E_max, n_steps)
mu       = 4000.0

residuals, models = pe.sweep(
    model_fn, E_values, sampler, V_fn,
    mu=mu, p=p, reset_every=25, n_epochs=2000,
)

pe.plot_loss_curve(E_values, residuals, title=f"Ex 3: p-Laplacian (p={p}), unit disk")
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn,
                              threshold=2.9, p=p)

for (E_approx, _), i in zip(eigenvalues[:2], idx[:2]):
    pe.plot_eigenfunction_disk(models[i], E_val=E_approx)

---
## Example 4 — p-Laplacian in the square [0,2]²  (p = 2.2)

In [ ]:
p = 2.2
E_min, E_max, n_steps = 0.0, 20.0, 50

sampler  = partial(pe.sample_square, x0=0.0, x1=2.0)
V_fn     = pe.V_zero
model_fn = lambda: pe.EigenFunctionNN(dim=2, boundary_fn=pe.boundary_square_2x2)

E_values = np.linspace(E_min, E_max, n_steps)

# μ adaptive: scales with E² to balance loss terms for higher eigenvalues
def mu_fn(E):
    return 1000.0 + E**2

residuals, models = [], []
model = model_fn()
for i, E in enumerate(E_values):
    if i > 0 and i % 20 != 0:
        new_model = model_fn()
        new_model.load_state_dict(model.state_dict())
        model = new_model
    else:
        model = model_fn()
    res, norm = pe.train_one(model, E, sampler, V_fn, mu=mu_fn(E), p=p, n_epochs=2500)
    residuals.append(res)
    models.append(model)
    print(f"  E={E:6.2f}  residual={res:.4e}  norm={norm:.2e}")

pe.plot_loss_curve(E_values, residuals, title=f"Ex 4: p-Laplacian (p={p}), [0,2]²")
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn,
                              threshold=3.5, p=p)

for (E_approx, _), i in zip(eigenvalues, idx):
    pe.plot_eigenfunction_square(models[i], E_val=E_approx, x0=0.0, x1=2.0)

---
## Example 5 — Laplacian in a triangular domain

$$-\Delta u = E u \quad \text{in } T, \qquad u=0 \text{ on } \partial T$$

where $T$ has vertices $(0,0)$, $(1,0)$, $(0,1)$.

Exact eigenvalues: $E_{mn} = \pi^2(m^2+n^2)$ (same as the unit square due to symmetry).

In [ ]:
EXACT = [5*np.pi**2, 10*np.pi**2, 13*np.pi**2, 17*np.pi**2]

E_min, E_max, n_steps = 0.0, 180.0, 100

sampler  = pe.sample_triangle
V_fn     = pe.V_zero
model_fn = lambda: pe.EigenFunctionNN(dim=2, boundary_fn=pe.boundary_triangle)

E_values = np.linspace(E_min, E_max, n_steps)
mu       = 15000.0

residuals, models = pe.sweep(
    model_fn, E_values, sampler, V_fn,
    mu=mu, reset_every=50, n_epochs=2000, n_samples=4000,
)

pe.plot_loss_curve(E_values, residuals, exact_eigenvalues=EXACT, ylim=1000,
                   title="Ex 5: Laplacian, triangle")
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn, threshold=6.0)

for (E_approx, _), i in zip(eigenvalues[:4], idx[:4]):
    pe.plot_eigenfunction_triangle(models[i], E_val=E_approx)

---
## Example 6 — Laplacian in the 4-D unit ball

Exact eigenvalues: $E_{mn} = y_{m,n}^2$ where $y_{m,n}$ is the $n$-th zero of $J_{m+1}$.

In [ ]:
EXACT_4D = [14.6819, 26.3746, 40.7064, 49.2184]

E_min, E_max, n_steps = 0.0, 53.0, 70

sampler  = pe.sample_ball_4d
V_fn     = pe.V_zero
model_fn = lambda: pe.EigenFunctionNN(dim=4, boundary_fn=pe.boundary_ball)

E_values = np.linspace(E_min, E_max, n_steps)
mu       = 200.0

residuals, models = pe.sweep(
    model_fn, E_values, sampler, V_fn,
    mu=mu, reset_every=60, n_epochs=2000,
)

pe.plot_loss_curve(E_values, residuals, exact_eigenvalues=EXACT_4D,
                   title="Ex 6: Laplacian, 4-D unit ball")
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn, threshold=2.0)

---
## Example 7 — Laplacian in the 3-D unit ball

Exact eigenvalues: $E_{mn} = j_{m,n}^2$ where $j_{m,n}$ is the $n$-th zero of $J_{m+1/2}$.

In [ ]:
EXACT_3D = [9.8696, 20.1907, 33.2251, 39.4784, 48.9737, 59.6946, 66.9489]

E_min, E_max, n_steps = 0.0, 75.0, 120

sampler  = pe.sample_ball_3d
V_fn     = pe.V_zero
model_fn = lambda: pe.EigenFunctionNN(dim=3, boundary_fn=pe.boundary_ball)

E_values = np.linspace(E_min, E_max, n_steps)
mu       = 400.0

residuals, models = pe.sweep(
    model_fn, E_values, sampler, V_fn,
    mu=mu, reset_every=50, n_epochs=2000,
)

pe.plot_loss_curve(E_values, residuals, exact_eigenvalues=EXACT_3D,
                   title="Ex 7: Laplacian, 3-D unit ball")
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn, threshold=1.3)

---
## Example 8 — Double-well Gaussian potential in the 3-D unit ball

$$V(x) = -e^{-\|x - a e_1\|^2/2\sigma^2} - e^{-\|x + a e_1\|^2/2\sigma^2}$$

In [ ]:
a, sigma = 1.0, 0.5

E_min, E_max, n_steps = 0.0, 75.0, 120

sampler  = pe.sample_ball_3d
V_fn     = partial(pe.V_double_well_gaussian, a=a, sigma=sigma)
model_fn = lambda: pe.EigenFunctionNN(dim=3, boundary_fn=pe.boundary_ball)

E_values = np.linspace(E_min, E_max, n_steps)
mu       = 400.0

residuals, models = pe.sweep(
    model_fn, E_values, sampler, V_fn,
    mu=mu, reset_every=50, n_epochs=2000,
)

pe.plot_loss_curve(E_values, residuals, title="Ex 8: Double-well Gaussian, 3-D ball")
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn, threshold=1.3)

---
## Example 9 — Compactly supported four-well potential in the 3-D unit ball

In [ ]:
E_min, E_max, n_steps = 0.0, 75.0, 120

sampler  = pe.sample_ball_3d
V_fn     = pe.V_four_well_bump
model_fn = lambda: pe.EigenFunctionNN(dim=3, boundary_fn=pe.boundary_ball)

E_values = np.linspace(E_min, E_max, n_steps)
mu       = 1000.0

residuals, models = pe.sweep(
    model_fn, E_values, sampler, V_fn,
    mu=mu, reset_every=50, n_epochs=2000, n_samples=2000,
)

pe.plot_loss_curve(E_values, residuals, title="Ex 9: Four-well bump potential, 3-D ball")
eigenvalues, idx = pe.refine(models, E_values, residuals, sampler, V_fn, threshold=1.3)